# 05 SQL Analysis

## Objective

Use SQL to analyze the Housing Affordability Stress Index dataset stored in SQLite.

## Database

- `housing_affordability.db`

## Main Questions

1. Which countries have the highest housing affordability stress in the latest year?
2. Which countries have the lowest housing affordability stress in the latest year?
3. How has average stress changed over time?
4. Which countries experienced the largest increase in stress?
5. Which countries have high or severe stress in the latest year?

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

DB_PATH = Path("../housing_affordability.db")
PROCESSED_DIR = Path("../data/processed")

print("Setup complete")

Setup complete


In [2]:
conn = sqlite3.connect(DB_PATH)

print("Connected to database")

Connected to database


In [3]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query, conn)

,name
0,housing_macro_country_year


In [4]:
query = """
SELECT *
FROM housing_macro_country_year
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,country_id,country_name,year,nominal_house_price_index,price_to_income_ratio,price_to_rent_ratio,real_house_price_index,rent_price_index,gdp_per_capita_constant_usd,gdp_per_capita_current_usd,...,rent_price_yoy_growth_pct,gdp_per_capita_yoy_growth_pct,house_price_income_gap,price_to_income_ratio_scaled,real_house_price_index_scaled,price_to_rent_ratio_scaled,urban_population_growth_pct_scaled,house_price_income_gap_scaled,housing_affordability_stress_score,housing_stress_category
0,AUS,Australia,2000,33.595673,67.822665,58.062251,47.961090,57.862362,45979.076565,21908.996802,...,NaN,NaN,0.629113,0.076037,0.260003,0.104041,0.456003,0.728783,23.090065,Low
1,AUS,Australia,2001,37.368512,69.777135,62.618848,51.543695,59.662722,46314.623530,19733.651010,...,3.111453,0.729782,6.740034,0.088434,0.279783,0.135394,0.483803,0.772478,25.360450,Low
2,AUS,Australia,2002,44.372389,81.271639,72.558473,59.617939,61.144035,47599.407929,20335.096019,...,2.482811,2.774036,12.890817,0.161339,0.324362,0.203785,0.479150,0.816459,30.787703,Low
3,AUS,Australia,2003,52.375657,91.792473,84.071694,68.972989,62.283510,48509.663218,23757.589847,...,1.863592,1.912325,13.779345,0.228068,0.376013,0.283004,0.479148,0.822812,36.062383,Low
4,AUS,Australia,2004,55.647000,91.233649,87.154344,72.246883,63.855978,50018.590448,30886.050095,...,2.524693,3.110570,1.636061,0.224524,0.394088,0.304215,0.465724,0.735983,35.811907,Low


## Query 1: Top 10 Countries by Housing Affordability Stress

This query ranks countries by their Housing Affordability Stress Score in the latest available year.

In [5]:
query = """
SELECT
    country_id,
    country_name,
    year,
    ROUND(housing_affordability_stress_score, 2) AS stress_score,
    housing_stress_category,
    ROUND(price_to_income_ratio, 2) AS price_to_income_ratio,
    ROUND(real_house_price_index, 2) AS real_house_price_index,
    ROUND(price_to_rent_ratio, 2) AS price_to_rent_ratio
FROM housing_macro_country_year
WHERE year = (
    SELECT MAX(year)
    FROM housing_macro_country_year
)
ORDER BY housing_affordability_stress_score DESC
LIMIT 10;
"""

top_10_sql = pd.read_sql_query(query, conn)

top_10_sql

,country_id,country_name,year,stress_score,housing_stress_category,price_to_income_ratio,real_house_price_index,price_to_rent_ratio
0,PRT,Portugal,2024,75.35,Severe,147.26,179.98,173.25
1,HUN,Hungary,2024,65.53,High,114.03,181.99,165.59
2,CAN,Canada,2024,65.08,High,136.19,144.16,135.82
3,NLD,Netherlands,2024,65.01,High,130.54,147.08,161.19
4,CZE,Czechia,2024,61.61,High,121.21,146.93,158.90
5,USA,United States,2024,61.42,High,128.28,154.04,133.31
6,GRC,Greece,2024,60.76,High,116.00,147.63,160.33
7,LTU,Lithuania,2024,58.64,Moderate,113.09,155.03,134.82
8,LUX,Luxembourg,2024,58.50,Moderate,121.52,131.34,144.07
9,SVN,Slovenia,2024,57.46,Moderate,120.95,154.47,117.43


In [6]:
top_10_sql.to_csv(
    PROCESSED_DIR / "sql_top_10_latest_housing_stress.csv",
    index=False
)

print("Saved SQL output:")
print(PROCESSED_DIR / "sql_top_10_latest_housing_stress.csv")

Saved SQL output:
..\data\processed\sql_top_10_latest_housing_stress.csv


### SQL Insight

The SQL ranking confirms the latest-year countries with the highest Housing Affordability Stress Scores. This output is useful for dashboard ranking views and executive summaries because it provides a concise view of the highest-pressure markets.

## Query 2: Average Housing Stress by Year

This query calculates the average, minimum, and maximum Housing Affordability Stress Score by year.

In [7]:
query = """
SELECT
    year,
    ROUND(AVG(housing_affordability_stress_score), 2) AS avg_stress_score,
    ROUND(MIN(housing_affordability_stress_score), 2) AS min_stress_score,
    ROUND(MAX(housing_affordability_stress_score), 2) AS max_stress_score,
    COUNT(DISTINCT country_id) AS countries_count
FROM housing_macro_country_year
GROUP BY year
ORDER BY year;
"""

yearly_stress_sql = pd.read_sql_query(query, conn)

yearly_stress_sql

,year,avg_stress_score,min_stress_score,max_stress_score,countries_count
0,2000,35.46,18.32,63.16,25
1,2001,36.27,18.30,64.23,26
2,2002,37.84,20.92,61.86,28
3,2003,39.58,22.25,60.94,28
4,2004,41.64,24.63,70.95,28
5,2005,43.92,26.37,69.42,33
6,2006,48.58,27.43,77.97,35
7,2007,51.90,26.32,87.89,38
8,2008,49.59,28.74,85.26,39
9,2009,45.40,31.56,73.09,40


In [8]:
yearly_stress_sql.to_csv(
    PROCESSED_DIR / "sql_yearly_housing_stress_trend.csv",
    index=False
)

print("Saved SQL output:")
print(PROCESSED_DIR / "sql_yearly_housing_stress_trend.csv")

Saved SQL output:
..\data\processed\sql_yearly_housing_stress_trend.csv


### SQL Insight

The yearly stress query summarizes how housing affordability pressure changed across the dataset over time. This table can support dashboard trend visuals and helps separate country-level rankings from broader time-series patterns.

## Query 3: Countries with the Largest Increase in Housing Stress

This query compares each country's first available Housing Affordability Stress Score with its latest available score to identify where affordability pressure increased the most.

In [9]:
query = """
WITH country_first_latest AS (
    SELECT
        country_id,
        country_name,
        MIN(year) AS first_year,
        MAX(year) AS latest_year
    FROM housing_macro_country_year
    GROUP BY country_id, country_name
),

country_scores AS (
    SELECT
        c.country_id,
        c.country_name,
        c.first_year,
        c.latest_year,
        f.housing_affordability_stress_score AS first_stress_score,
        l.housing_affordability_stress_score AS latest_stress_score,
        l.housing_affordability_stress_score - f.housing_affordability_stress_score AS stress_score_change
    FROM country_first_latest c
    JOIN housing_macro_country_year f
        ON c.country_id = f.country_id
       AND c.first_year = f.year
    JOIN housing_macro_country_year l
        ON c.country_id = l.country_id
       AND c.latest_year = l.year
)

SELECT
    country_id,
    country_name,
    first_year,
    latest_year,
    ROUND(first_stress_score, 2) AS first_stress_score,
    ROUND(latest_stress_score, 2) AS latest_stress_score,
    ROUND(stress_score_change, 2) AS stress_score_change
FROM country_scores
ORDER BY stress_score_change DESC
LIMIT 10;
"""

stress_deterioration_sql = pd.read_sql_query(query, conn)

stress_deterioration_sql

,country_id,country_name,first_year,latest_year,first_stress_score,latest_stress_score,stress_score_change
0,CAN,Canada,2000,2024,21.54,65.08,43.54
1,NZL,New Zealand,2000,2024,18.32,54.51,36.19
2,AUS,Australia,2000,2024,23.09,56.43,33.34
3,CHE,Switzerland,2000,2024,29.39,56.10,26.71
4,ESP,Spain,2000,2024,33.08,57.25,24.17
5,SWE,Sweden,2000,2024,19.03,42.95,23.92
6,CHL,Chile,2002,2024,28.79,51.38,22.58
7,NOR,Norway,2000,2024,26.39,48.60,22.21
8,GBR,United Kingdom,2000,2024,25.87,47.51,21.64
9,LUX,Luxembourg,2007,2024,37.35,58.50,21.15


In [10]:
stress_deterioration_sql.to_csv(
    PROCESSED_DIR / "sql_largest_housing_stress_deterioration.csv",
    index=False
)

print("Saved SQL output:")
print(PROCESSED_DIR / "sql_largest_housing_stress_deterioration.csv")

Saved SQL output:
..\data\processed\sql_largest_housing_stress_deterioration.csv


### SQL Insight

The deterioration query identifies countries where affordability stress increased the most over the available period. This is different from the latest-year ranking because it focuses on change over time rather than the current level of stress.

## Query 4: Latest-Year High-Stress Countries

This query identifies countries in the latest available year that are categorized as High or Severe housing affordability stress.

In [11]:
query = """
SELECT
    country_id,
    country_name,
    year,
    ROUND(housing_affordability_stress_score, 2) AS stress_score,
    housing_stress_category,
    ROUND(price_to_income_ratio, 2) AS price_to_income_ratio,
    ROUND(price_to_rent_ratio, 2) AS price_to_rent_ratio,
    ROUND(real_house_price_index, 2) AS real_house_price_index
FROM housing_macro_country_year
WHERE year = (
    SELECT MAX(year)
    FROM housing_macro_country_year
)
AND housing_stress_category IN ('High', 'Severe')
ORDER BY housing_affordability_stress_score DESC;
"""

high_stress_sql = pd.read_sql_query(query, conn)

high_stress_sql

,country_id,country_name,year,stress_score,housing_stress_category,price_to_income_ratio,price_to_rent_ratio,real_house_price_index
0,PRT,Portugal,2024,75.35,Severe,147.26,173.25,179.98
1,HUN,Hungary,2024,65.53,High,114.03,165.59,181.99
2,CAN,Canada,2024,65.08,High,136.19,135.82,144.16
3,NLD,Netherlands,2024,65.01,High,130.54,161.19,147.08
4,CZE,Czechia,2024,61.61,High,121.21,158.90,146.93
5,USA,United States,2024,61.42,High,128.28,133.31,154.04
6,GRC,Greece,2024,60.76,High,116.00,160.33,147.63


In [12]:
high_stress_sql.to_csv(
    PROCESSED_DIR / "sql_latest_high_stress_countries.csv",
    index=False
)

print("Saved SQL output:")
print(PROCESSED_DIR / "sql_latest_high_stress_countries.csv")

Saved SQL output:
..\data\processed\sql_latest_high_stress_countries.csv


In [13]:
conn.close()
print("Database connection closed")

Database connection closed


## SQL Analysis Summary

The SQL analysis produced dashboard-ready outputs for:

- Latest-year top 10 stress rankings
- Yearly average stress trends
- Countries with the largest stress score increase
- Latest-year High and Severe stress countries

These queries demonstrate SQL skills such as filtering, aggregation, joins, common table expressions, subqueries, ordering, and dashboard table preparation.